<a href="https://colab.research.google.com/github/matthewhawksby/colabnotebooks/blob/main/Localization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mne
import os
import sys
import pandas as pd
import numpy as np
import mne
import matplotlib.pyplot as plt
import glob
from pathlib import Path


from google.colab import drive
drive.mount('/content/drive')

import helper
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src/data/"
TSV_PATH = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/stimuli/demo/*.tsv"
SRC_DIR = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src"
sys.path.append(SRC_DIR)

import helper
import importlib
importlib.reload(helper)

subjects_dir = Path("/content/drive/MyDrive/fsaverage")
mne.datasets.fetch_fsaverage(subjects_dir=subjects_dir, verbose=True)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
0 files missing from root.txt in /content/drive/MyDrive/fsaverage
0 files missing from bem.txt in /content/drive/MyDrive/fsaverage/fsaverage


PosixPath('/content/drive/MyDrive/fsaverage/fsaverage')

In [2]:
def load_all_csvs(csv_dir="/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src/data"):
    csv_files = sorted(glob.glob(os.path.join(csv_dir, "M0*_main_*.csv")))
    df_map = {}

    for path in csv_files:
        filename = os.path.basename(path)
        participant = filename.split("_")[0]  # M001, M002, etc.
        if participant not in df_map:
            df_map[participant] = []
        df_map[participant].append(pd.read_csv(path))

    # Concatenate each participant's CSVs
    for participant in df_map:
        df_map[participant] = pd.concat(df_map[participant], ignore_index=True)

    return df_map  # { 'M001': DataFrame, 'M002': DataFrame, ... }

csv_data = load_all_csvs()



In [3]:
### Why does M001 have more columns?

for participant, df in csv_data.items():
    print(f"{participant}: {df.shape}")
    # Compare column names across all participant DataFrames


def compare_column_names(csv_data):
    participant_cols = {p: set(df.columns) for p, df in csv_data.items()}

    # Print column counts
    for p, cols in participant_cols.items():
        print(f"{p}: {len(cols)} columns")

    print("\n🔍 Unique column differences:")

    participants = list(participant_cols.keys())
    for i in range(len(participants)):
        for j in range(i + 1, len(participants)):
            p1, p2 = participants[i], participants[j]
            only_in_p1 = participant_cols[p1] - participant_cols[p2]
            only_in_p2 = participant_cols[p2] - participant_cols[p1]

            if only_in_p1 or only_in_p2:
                print(f"\n🔺 {p1} vs {p2}:")
                if only_in_p1:
                    print(f"  - Only in {p1}: {sorted(only_in_p1)}")
                if only_in_p2:
                    print(f"  - Only in {p2}: {sorted(only_in_p2)}")
            else:
                print(f"\n✅ {p1} and {p2} have identical columns.")

compare_column_names(csv_data)



M001: (675, 67)
M002: (658, 61)
M003: (657, 61)
M001: 67 columns
M002: 61 columns
M003: 61 columns

🔍 Unique column differences:

🔺 M001 vs M002:
  - Only in M001: ['Unnamed: 12', 'Unnamed: 53', 'Unnamed: 64', 'key_resp.duration', 'key_resp.rt', 'trials.key_resp.duration', 'trials.key_resp.rt']
  - Only in M002: ['Unnamed: 60']

🔺 M001 vs M003:
  - Only in M001: ['Unnamed: 12', 'Unnamed: 53', 'Unnamed: 64', 'key_resp.duration', 'key_resp.rt', 'trials.key_resp.duration', 'trials.key_resp.rt']
  - Only in M003: ['Unnamed: 60']

✅ M002 and M003 have identical columns.


In [4]:
def load_meg_blocks(participant_id, base_path="/content/drive/MyDrive/Colab Notebooks/MNE/MNE_data"):
    block_path = os.path.join(base_path, f"m_{participant_id[1:]}", "250414")
    fif_files = sorted(glob.glob(os.path.join(block_path, f"{participant_id}_block*_raw.fif")))

    if not fif_files:
        raise FileNotFoundError(f"No .fif files found for {participant_id} in {block_path}")

    raws = [mne.io.read_raw_fif(f, preload=True, verbose=False) for f in fif_files]

    # Override incompatible dev_head_t to allow concatenation
    for raw in raws[1:]:
        raw.info['dev_head_t'] = raws[0].info['dev_head_t']

    raw_combined = mne.concatenate_raws(raws, preload=True)
    return raw_combined



In [5]:
def get_stg_channels(raw):
    """
    Uses MNE’s fsaverage source space to approximate STG sensor projection
    Returns list of MEG channel names closest to the superior temporal gyrus
    """
    subjects_dir = mne.datasets.fetch_fsaverage(verbose=True)
    subject = "fsaverage"

    # Setup source space on fsaverage
    src = mne.setup_source_space(subject, spacing='oct6', subjects_dir=subjects_dir, add_dist=False, verbose=False)

    # Load standard MRI/head model for fsaverage
    model = mne.make_forward_solution(info=raw.info, trans='fsaverage', src=src,
                                      bem='fsaverage', meg=True, eeg=False, verbose=False)

    # Project source points to sensor space
    stc = mne.stc_near_sensors(model, dist=0.03, return_info=True)
    stg_indices = stc[0].vertices[0]  # get vertex indices from lh

    # Get sensors sensitive to STG
    picked = mne.pick_types_forward(model, meg=True)
    ch_names = [model['sol']['row_names'][i] for i in picked]

    return list(set(ch_names))


In [6]:
def compare_hearing_vs_silent(raw, df, pre=0.5, post=1.0):
    """
    Compare brain activity during sound vs silence.
    - raw: mne.io.Raw
    - df: DataFrame with 'sound_1.started' column (stimulus onset)
    """
    # --- Filter to remove drift and noise
    raw_filt = raw.copy().filter(1., 40., fir_design='firwin')

    # --- Extract sound onset times
    onsets = df['sound_1.started'].dropna().values
    print(onsets[1])
    durations = [post] * len(onsets)
    print(durations[1])
    descs = ['hearing'] * len(onsets)
    print(descs[1])

    # --- Mark hearing windows
    annots = mne.Annotations(onset=onsets, duration=durations, description=descs)
    raw_filt.set_annotations(annots)

    # --- Create events from annotations
    events, event_id = mne.events_from_annotations(raw_filt)

    # --- Epoch around sound (e.g. 0.5s before to 1s after)
    epochs = mne.Epochs(raw_filt, events, event_id=event_id,
                        tmin=-pre, tmax=post, baseline=None, preload=True)

    # --- All MEG sensors only
    epochs_meg = epochs.pick_types(meg=True)

    # --- RMS over channels: shape (n_epochs, n_times)
    data = epochs_meg.get_data()
    rms_per_epoch = np.sqrt(np.square(data).mean(axis=1))  # (epochs, time)

    # --- Mean activity over epochs and channels (simplified)
    avg_rms = rms_per_epoch.mean(axis=0)
    times = epochs.times

    # --- Plot
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,4))
    plt.plot(times, avg_rms)
    plt.axvline(0, linestyle='--', color='black', label='Sound Onset')
    plt.xlabel("Time (s)")
    plt.ylabel("RMS Activity (a.u.)")
    plt.title("MEG Activity (All Sensors, Hearing Only)")
    plt.legend()
    plt.grid()
    plt.show()



In [17]:
def print_sample_meg_values(raw, df, n_samples=3, duration_sec=0.2):
    # Pick only MEG channels
    picks = mne.pick_types(raw.info, meg=True)

    sfreq = raw.info['sfreq']
    duration = int(duration_sec * sfreq)

    print("\n🎧 Sample HEARING segments (MEG only):")
    hearing_trials = df[df['sound_1_onset'] > 0].sample(n=n_samples)
    for i, row in hearing_trials.iterrows():
        start = int(row['sound_1_onset'] * sfreq)
        stop = start + duration
        segment, _ = raw[picks, start:stop]
        print(f"Trial {i} - shape: {segment.shape}, values:\n{segment[:, :5]}")

    print("\n🤫 Sample SILENT segments (MEG only):")
    for i, row in hearing_trials.iterrows():
        start = int((row['sound_1_onset'] - 0.5) * sfreq)
        if start < 0:
            continue
        stop = start + duration
        segment, _ = raw[picks, start:stop]
        print(f"Trial {i} - shape: {segment.shape}, values:\n{segment[:, :5]}")
    print(f"Trial {i}: onset={row['sound_1.started']}, start={start}, stop={stop}, raw.n_times={raw.n_times}")




In [18]:
for pid in ['M001', 'M002', 'M003']:
    print(f"\n🔎 Participant {pid}")
    raw = load_meg_blocks(pid)
    df = csv_data[pid]
    print_sample_values(raw, df)
    print("→ Running with ALL sensors")

    #compare_hearing_vs_silence(raw, df)

    # Optional: Use auditory (STG) subset
    try:
        stg_channels = get_stg_channels(raw, subjects_dir="/content/drive/MyDrive/fsaverage")
        print(f"→ Running with STG sensors ({len(stg_channels)})")
        compare_hearing_vs_silence(raw, df, channels=stg_channels)
    except:
        print("⚠️ Could not isolate STG sensors — skipping subset")




🔎 Participant M001

🎧 Sample HEARING segments:
Trial 305 - shape: (328, 400), sample values: [[ 1.75842766e-04  1.75007805e-04  1.75842766e-04  1.77345695e-04
   1.79015617e-04]
 [-3.96995920e-04 -3.99277506e-04 -3.99277506e-04 -3.99979532e-04
  -4.00857065e-04]
 [ 2.72640674e-05  2.84494617e-05  2.43852529e-05  2.72640674e-05
   3.72552474e-05]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [ 7.93600000e+03  7.93600000e+03  7.93600000e+03  7.93600000e+03
   7.93600000e+03]]
Trial 5 - shape: (328, 400), sample values: [[ 2.02060538e-04  2.01058585e-04  1.99555655e-04  1.98720694e-04
   1.97384757e-04]
 [-6.79210526e-05 -6.86230790e-05 -6.84475724e-05 -6.98516252e-05
  -7.31862505e-05]
 [ 5.89310277e-05  5.99470799e-05  6.13018162e-05  5.89310277e-05
   5.43587928e-05]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 